# AIPI 510 Week 7 Interactive: MoodMate App - Effect Size & Power Analysis 
*AIPI Sourcing Data — Week 7: Effect Size Files*

- ** Claim:** “Users who log their emotions daily using our app report a 25% lower chance of depressive symptoms after 30 days.”
- ** 1. Null and Alternative Hypotheses:** Section 2
- ** 2. Statistical Test:** Section 2
- ** 3. Explore Literature to Find Effect Size:** Section 3
- ** 4. Perform Power Analysis:** Section 4
- Provide Rationale: Explanations in Sections 3 and 5
- Deliverable: PR of your Jupyter notebook with all analyses and justification into the week7 branch in the subrepo effect-size-files


## Claim and Study Setup
This section describes the design and measurement for the chosen claim.

Outcome variable: Presence of depressive symptoms (yes=1, no=0).
Design: Randomized trial comparing MoodMate users vs. Control group.
Measure: Binary variable after 30 days - symptoms: yes/no  (binary)

We are testing if MoodMate reduces the proportion of people showing depressive symptoms.

## What are your null and alternative hypotheses?
Null Hypothesis: p_MoodMate − p_Control = 0  (no difference)

Alternative Hypothesis: p_MoodMate − p_Control < 0  (MoodMate lowers depression rates)

Test used: Two-sample z-test for proportions (choosing this because of how the data is binary).
We’ll also calculate Cohen’s h, the effect size for proportions.

## Explore literature to find effect siz
From meta analyses of digital mental health interventions it appears that the typical standardized effects range around small to moderate (which is about ≈ 0.25–0.45).

The company’s claim of a 25% relative reduction in depressive symptoms is consistent with that range, so we’ll use a 25% relative risk reduction for our target effect.

I will test several baseline risks (20%, 30%, 40%) to see how sample size changes.

In [ ]:
# --- Initial Import for necessary libraries ---
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# For power and effect size calculations
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize

## Perform power analysis w/ effect size
This step answers How many participants do we need

I'll calculate Cohen’s h for each baseline risk then we'll find the required number of participants per group for 80% power .05 (one-sided test, since we expect MoodMate to reduce depression).

In [ ]:
# tests
alpha = 0.05   # significance level
power = 0.80   # desired power
alternative = 'larger'  # one sided test (MoodMate is expected to reduce depression)

control_risks = [0.20, 0.30, 0.40]   #  control event rates
rrr = 0.25                           # rrr is the relative risk reduction , here it is a 25% relative risk reduction (the company claim)

# effect sizes and sample sizes 
rows = []
analysis = NormalIndPower()

for p_c in control_risks:
    p_t = p_c * (1 - rrr)  # treatment risk after 25% reduction
    es_h = proportion_effectsize(p1=p_c, p2=p_t)  # Cohen's h
    n_per_group = analysis.solve_power(effect_size=es_h, alpha=alpha, power=power, ratio=1.0, alternative=alternative)
    rows.append({
        'Control Risk': p_c,
        'Treatment Risk (25% RRR)': round(p_t, 3),
        'Cohen_h': round(es_h, 3),
        'n_per_group (ceil)': int(np.ceil(n_per_group)),
        'total_n (ceil)': int(2*np.ceil(n_per_group))
    })

# Displaying results as a simple table
pd.DataFrame(rows)

### Interpretation
- Pick the largest sample size (from the 40% baseline) to stay conservative.
- This is to ensures the study has enough power even if depression rates are higher.

## Sample Analysis 
Since the excel sheet of data isn't available, I'll be using simulated data for the remainder of the test here. This is how you’d test your actual data once collected. To start, we’ll simulate results for 200 participants per group to demonstrate the process.The same code will work when you replace the simulated data with your actual study data.

In [ ]:
np.random.seed(7)  # setting a seed is to force the same starting point at the random number generator in NumPy, this will maintain the same numbers for reruns and testing
n_per_group_planned = 200
p_control = 0.30
p_treat = p_control * (1 - 0.25)  # thhe 25% reduction in risk

# --- Generate binary data with 1 = depressive symptoms, and 0 equals = no symptoms ---
control = np.random.binomial(1, p_control, size=n_per_group_planned)
treat = np.random.binomial(1, p_treat, size=n_per_group_planned)

# --- Computing proportions and run two proportion z-test ---
x1 = treat.sum(); x2 = control.sum()
n1 = len(treat); n2 = len(control)
p1 = x1/n1; p2 = x2/n2

print('Counts:', {'treat_yes': int(x1), 'control_yes': int(x2)})
print('Proportions:', {'p_treat': round(p1,3), 'p_control': round(p2,3)})

count = np.array([x1, x2])
nobs = np.array([n1, n2])
stat, pval = proportions_ztest(count, nobs, alternative='smaller')
print('Z test:', {'z_stat': round(stat,3), 'p_value': round(float(pval),4)})

# --- Cohen's h as effect size---
h = proportion_effectsize(p1=p2, p2=p1)
print('Cohen_h:', round(abs(h),3))

### Code Explanation
- We simulate data using a binomial model (1 = depression, 0 = none).
- Then we calculate the sample proportions for each group.
- The z-test is to checks whether MoodMate’s proportion is significantly lower.
- The p-value tells you if the difference is statistically significant.
- The Cohen’s h is for describing how large the difference is (aka effect size).

In [ ]:
# --- visualization ---
labels = ['Control', 'MoodMate']
props = [p2, p1]
plt.bar(labels, props, color=['gray', 'skyblue'])
plt.title('Proportion with Depressive Symptoms at Day 30')
plt.ylabel('Proportion')
plt.ylim(0, max(props)+0.1)
plt.show()

### Final Conclusion: This analysis showcases a simple and hypothesis driven study design to test the MoodMate App claim that daily emotion logging reduces depressive symptoms. Using our hypothesis and  assumptions and developing a literature based effect size, we conducted a power analysis to estimate sample size needs and demonstrated how to test the results once the data is collected. The results show that if MoodMate users have around a ~25% lower rate of depressive symptoms, the study would require about ~ 150–200 participants per group to detect that effect with 80% power. 
